# Simple Seq2Seq From Scratch (PyTorch)

This notebook builds a minimal encoder-decoder Seq2Seq model from scratch and trains it on a toy task: reversing short character sequences.

You can adapt the same structure for translation, summarization, or other sequence generation tasks.

In [ ]:
# If PyTorch is missing, uncomment the next line and run this cell once:
# %pip install torch

import random
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

print('PyTorch:', torch.__version__)

PyTorch: 2.10.0+cu128


In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [ ]:
# 1) Build a toy dataset
# Input:  random string, e.g. 'cat'
# Output: reversed string, e.g. 'tac'

ALPHABET = list('abcdefghijklmnopqrstuvwxyz')

def random_word(min_len=3, max_len=8):
    n = random.randint(min_len, max_len)
    return ''.join(random.choice(ALPHABET) for _ in range(n))

pairs = []
for _ in range(4000):
    src = random_word()
    tgt = src[::-1]
    pairs.append((src, tgt))

train_pairs = pairs[:3600]
val_pairs = pairs[3600:]

print('train size:', len(train_pairs), 'val size:', len(val_pairs))
print('example:', train_pairs[0])

train size: 3600 val size: 400
example: ('daxihhex', 'xehhixad')


In [ ]:
# 2) Tokenization / vocabulary
PAD = '<pad>'
SOS = '<sos>'
EOS = '<eos>'

itos = [PAD, SOS, EOS] + ALPHABET
stoi = {tok: i for i, tok in enumerate(itos)}

PAD_ID = stoi[PAD]
SOS_ID = stoi[SOS]
EOS_ID = stoi[EOS]

def encode_text(text):
    return [stoi[ch] for ch in text]

def decode_ids(ids):
    out = []
    for i in ids:
        tok = itos[i]
        if tok == EOS:
            break
        if tok not in {PAD, SOS}:
            out.append(tok)
    return ''.join(out)

VOCAB_SIZE = len(itos)
print('vocab size:', VOCAB_SIZE)

vocab size: 29


In [ ]:
# 3) Dataset + collate
class Seq2SeqDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_ids = encode_text(src) + [EOS_ID]
        tgt_in = [SOS_ID] + encode_text(tgt)
        tgt_out = encode_text(tgt) + [EOS_ID]
        return torch.tensor(src_ids), torch.tensor(tgt_in), torch.tensor(tgt_out)

def pad_1d(seqs, pad_id=PAD_ID):
    max_len = max(s.size(0) for s in seqs)
    out = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :s.size(0)] = s
    return out

def collate_fn(batch):
    src, tgt_in, tgt_out = zip(*batch)
    return pad_1d(src), pad_1d(tgt_in), pad_1d(tgt_out)

train_loader = DataLoader(Seq2SeqDataset(train_pairs), batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(Seq2SeqDataset(val_pairs), batch_size=64, shuffle=False, collate_fn=collate_fn)

src_b, tgt_in_b, tgt_out_b = next(iter(train_loader))
print(src_b.shape, tgt_in_b.shape, tgt_out_b.shape)

In [ ]:
# 4) Encoder + Decoder + Seq2Seq
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        emb = self.embedding(src)
        _, hidden = self.gru(emb)
        return hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, hidden):
        # x: [batch] (one token at current step)
        x = x.unsqueeze(1)
        emb = self.embedding(x)
        out, hidden = self.gru(emb, hidden)
        logits = self.fc(out.squeeze(1))
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, sos_id):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sos_id = sos_id

    def forward(self, src, tgt_in, teacher_forcing_ratio=1.0):
        batch_size = src.size(0)
        tgt_len = tgt_in.size(1)
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src.device)
        hidden = self.encoder(src)

        x = torch.full((batch_size,), self.sos_id, dtype=torch.long, device=src.device)

        for t in range(tgt_len):
            logits, hidden = self.decoder(x, hidden)
            outputs[:, t, :] = logits

            use_teacher = random.random() < teacher_forcing_ratio
            top1 = logits.argmax(1)
            x = tgt_in[:, t] if use_teacher else top1

        return outputs

In [ ]:
# 5) Train
@dataclass
class Config:
    emb_dim: int = 64
    hid_dim: int = 128
    lr: float = 1e-3
    epochs: int = 12
    teacher_forcing_ratio: float = 0.8

cfg = Config()

encoder = Encoder(VOCAB_SIZE, cfg.emb_dim, cfg.hid_dim, PAD_ID)
decoder = Decoder(VOCAB_SIZE, cfg.emb_dim, cfg.hid_dim, PAD_ID)
model = Seq2Seq(encoder, decoder, SOS_ID).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0

    for src, tgt_in, tgt_out in loader:
        src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(src, tgt_in, teacher_forcing_ratio=cfg.teacher_forcing_ratio if train else 0.0)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

for epoch in range(1, cfg.epochs + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')

In [ ]:
# 6) Inference (greedy decoding)
@torch.no_grad()
def predict(text, max_steps=20):
    model.eval()

    src = torch.tensor([encode_text(text) + [EOS_ID]], device=DEVICE)
    hidden = model.encoder(src)

    x = torch.tensor([SOS_ID], device=DEVICE)
    out_ids = []

    for _ in range(max_steps):
        logits, hidden = model.decoder(x, hidden)
        x = logits.argmax(1)
        token_id = x.item()
        if token_id == EOS_ID:
            break
        out_ids.append(token_id)

    return decode_ids(out_ids + [EOS_ID])

tests = ['cat', 'finance', 'market', 'hello', 'abcxyz']
for t in tests:
    print(f'input={t:10s} pred={predict(t):10s} target={t[::-1]}')

## Next steps

- Replace toy string-reversal data with real parallel text (e.g., sentence pairs).
- Add an attention mechanism to avoid strict fixed-length bottleneck.
- Use BLEU/ROUGE-style metrics for real NLP tasks.